# **Automated Extraction and AI Analysis of Infrastructure Assets from Multi-Source Imagery Using `rAPIdtools`**

## <u>Part 1: Automated Extraction of Assets from RAPID Imagery</u>
In this first step, we automatically identify and map all buildings within the disaster zone without dependence on pre-existing GIS databases. By integrating high-resolution aerial imagery with Meta's Segment Anything Model 3 (SAM 3), we construct a foundational asset inventory from the ground up. Although this example emphasizes buildings, the process is applicable to any physical asset discernible in aerial imagery.

### **Workflow Overview:**

* **Define the Region of Interest:**
  A post-disaster RAPID drone raster is loaded to establish the precise geographic bounding box for analysis.
* **Synthesize Baseline Imagery:**
  The `BingOrthomosaicExtractor` is deployed to automatically download, stitch, and georeference a high-resolution, pre-disaster orthomosaic for the target zone.
* **Preliminary AI Extraction:**
  The `SAM3OrthoFeatureExtractor` is applied across the entire synthesized map, prompting the vision model to detect and mask preliminary building roofs.
* **Geometric Regularization:**
  Targeted image crops are extracted around these initial detections and processed with the `BuildingRegularizer`. This step refines the raw AI masks by merging overlapping polygons, healing fragmented shapes, and enforcing clean property boundaries.

> **Result:** *A GIS-ready GeoJSON of physical assets, prepared for downstream multi-temporal damage and recovery assessments.*

### Import required rAPIdtools methods and packages

In [1]:
from rapidtools import (
    AerialImageryExtractor,
    BingOrthomosaicExtractor,
    BoundingBox,
    BuildingRegularizer,
    Gemma4AssetAnalyzer,
    MapillaryImageExtractor,
    SAM3OrthoFeatureExtractor,
    Pipeline,
    PolygonRegion,
    PhysicalAssetCollection,
    download_dataset
)

from pathlib import Path

### Download a UW RAPID post-event raster for the 2025 Eaton Fire in Altadena, CA and determine its bounds
**Downloading the reference data**
```python
[raster_path] = download_dataset('eaton_patch1')
```
* **The action:** We use the built-in `download_dataset` utility to grab a sample post-fire drone map (an orthomosaic) of the Eaton Fire.
* **The syntax:** Because `download_dataset` always returns a list of file paths, we use Python's list-unpacking syntax (the brackets `[...]` around `raster_path`) to neatly assign that single file path directly to our variable.

**Defining the spatial extent**
```python
region_of_interest = BoundingBox.from_raster(raster_path)
```
* **The action:** Instead of manually typing out latitude and longitude coordinates, we use the `BoundingBox.from_raster()` helper. This `BoundingBox` will now act as the geographic fence for the rest of our pipeline!

In [2]:
[raster_path] = download_dataset('eaton_patch1')
region_of_interest = BoundingBox.from_raster(raster_path)

2026-05-27 07:25:51- INFO: Preparing to download 1 file(s) across 1 dataset(s)...


2026-05-27 07:26:47- INFO: Successfully secured 1/1 files.


### Synthesize a pre-disaster baseline map using Bing Maps tiles
**Initialization of the image extractor**
```python
bing_extractor = BingOrthomosaicExtractor(max_workers=10)
```
* **The action:** The extractor is configured to retrieve highly detailed aerial imagery from Bing Maps.
* **The performance:** Setting `max_workers=10` initiates a multi-threaded connection pool, enabling simultaneous downloading of multiple image tiles and significantly reducing download times.

**Orthomosaic stitching process**
```python
bing_raster_path = bing_extractor(
    region=region_of_interest,
    output_path='eaton_patch1_bing.tiff'
)
```
* **The action:** The `region_of_interest` boundary is specified to restrict the download to the area corresponding precisely to the post-disaster RAPID drone map coverage.
* **The output:** The extractor automatically aligns and merges the individual tiles, saving them as a unified  `eaton_patch1_bing.tiff` file. The Coordinate Reference System (CRS) metadata required for subsequent AI processing is also automatically embedded.

In [3]:
bing_extractor = BingOrthomosaicExtractor(max_workers=10)
bing_raster_path = bing_extractor(
    region=region_of_interest,
    output_path='eaton_patch1_bing.tiff'
)

2026-05-27 07:26:47- INFO: Stitching 1991x1355 px synthetic orthomosaic at zoom 19...
2026-05-27 07:26:48- INFO: Saving GeoTIFF to: /home/bacetiner/Desktop/workshop_demo/eaton_patch1_bing.tiff


### Extract preliminary building footprints using Meta's Segment Anything Model (SAM 3)
**Configuring the AI extractor**
```python
sam_extractor = SAM3OrthoFeatureExtractor(
    prompt='building roof',
    patch_size=50,          
    unit='meters',           
    overlap_ratio=0.25,      
    batch_size=4,            
    threshold=0.50,
    mask_threshold=0.40,
)
```
* **The action:** The feature extractor is initialized, and the zero-shot SAM 3 model is prompted to specifically identify any `'building roof'`.
* **What is under the hood?:** Due to the large size of high-resolution orthomosaics, the tool automatically divides the map into 50-by-50-meter patches with 25% overlap, processing them in batches of 4 to accommodate GPU memory constraints.
* **Understanding the thresholds:**
  * `threshold=0.50` *(Object Confidence)*:  This parameter specifies the minimum confidence required for the model to classify a region as a building roof. A threshold of 50% enables the detection of most structures while effectively excluding noise such as flat driveways or large vehicles.
  * `mask_threshold=0.40` *(Pixel Boundary)*: SAM 3 produces a probability heat map for each pixel. Setting this threshold slightly below 50% increases inclusivity, allowing the mask to extend to the edges of rooflines, even when partially obscured by shadows or vegetation.

**Executing the map sweep**
```python
prelim_collection = sam_extractor(bing_raster_path)
```
* **The action:** The synthesized pre-disaster Bing map is pulled into the extractor. The tool systematically sweeps the image, converting AI pixel masks into georeferenced Shapely polygons for each detected building.

**Saving the preliminary results**
```python
_ = prelim_collection.to_geojson('eaton_patch1_buildings_preliminary.geojson')
```
* **The action:** The initial, raw AI detections are exported to a GeoJSON file. As zero-shot AI models may generate overlapping regions or irregular boundaries, this output serves as a preliminary inventory prior to geometric regularization in the subsequent step.

In [4]:
sam_extractor = SAM3OrthoFeatureExtractor(
    prompt='building roof',
    patch_size=50,
    unit='meters',
    overlap_ratio=0.25,
    batch_size=4,
    threshold=0.50,
    mask_threshold=0.40,
)

prelim_collection = sam_extractor(bing_raster_path)

_ = prelim_collection.to_geojson('eaton_patch1_buildings_preliminary.geojson')

2026-05-27 07:26:48- INFO: Initializing SAM3OrthoFeatureExtractor for 'building roof' at scale: 50 meters with threshold 0.5...
2026-05-27 07:26:48- INFO: Loading processor and weights for facebook/sam3 (this may take a minute)...


Loading weights:   0%|          | 0/1468 [00:00<?, ?it/s]

2026-05-27 07:27:00- INFO: SAM 3 model facebook/sam3 loaded successfully.


Scanning for 'building roof': 126it [00:36,  3.41it/s]


2026-05-27 07:27:38- INFO: Extracted 815 raw polygons. Merging overlaps...
2026-05-27 07:27:38- INFO: Extraction complete. Yielded 276 unique assets.


### Extract buffered image crops to provide spatial context for geometric regularization
**Configuring the aerial image extractor**
```python
extractor = AerialImageryExtractor(
    dataset=bing_raster_path,
    save_directory='output/building_crops',
    buffer_asset='20 m',
    force_square_image=True,
)
```
* **The action:** The extractor is configured to generate specific, localized image patches from the pre-disaster Bing map for each building detected in the previous step.
* **The spatial context:** Setting `buffer_asset='20 m'` instructs the extractor to add an additional 20 meters to the raw AI footprint. This visual padding is essential, as it provides the regularization model with sufficient context to distinguish the roofline from adjacent areas, such as the backyard.
* **The formatting:** Setting `force_square_image=True` mathematically adjusts the bounding boxes to ensure that each cropped image is a perfect square, which is the optimal aspect ratio for Vision-Language Models.

**Attaching imagery to the inventory**
```python
assets_with_images = extractor(prelim_collection)
```
* **The action:** The preliminary asset collection is provided to the extractor, which iterates through the geometry, extracts high-resolution JPEGs, saves them to disk, and automatically attaches these new image records to the corresponding `PhysicalAsset` objects in the collection.

In [5]:
extractor = AerialImageryExtractor(
    dataset=bing_raster_path,
    save_directory='output/building_crops',
    buffer_asset='20 m',
    force_square_image=True,
)

assets_with_images = extractor(prelim_collection)

2026-05-27 07:27:39- INFO: Found 1 raster(s). Images will be saved to: /home/bacetiner/Desktop/workshop_demo/output/building_crops
2026-05-27 07:27:39- INFO: Processing raster: eaton_patch1_bing.tiff
2026-05-27 07:27:39- INFO: Found 276/276 assets within the current raster's extent.


Extracting from eaton_patch1_bing.tiff: 100%|█| 276/276 [00:02<00:00, 120.76it/s

2026-05-27 07:27:41- INFO: Finished processing all rasters. Extracted aerial imagery for a total of 276 assets.


### Refine and regularize building footprints into clean, GIS-ready polygons
**Initializing the regularizer**
```python
regularizer = BuildingRegularizer(batch_size=8)
```
* **The action:** Configure the `BuildingRegularizer` to process assets in GPU-optimized batches of eight.

**Executing geometric regularization**
```python
final_buildings = regularizer(assets_with_images)
```
* **The action:** Feed the collection of preliminary assets and their centered, buffered image crops into the regularizer to refine the geometries.
* **What is under the hood?:** This is where the heavy lifting happens. The regularizer performs highly context-aware segmentation on the zoomed-in image crops. It then mathematically refines the output by dissolving overlapping AI duplicates, bridging fragments split by shadows or tree branches, and performing straight-edge slicing to resolve overlaps between neighboring properties. Finally, it filters out degenerate shapes and automatically tags every valid asset as a `'building'`.

**3. Saving the finalized asset inventory**
```python
_ = final_buildings.to_geojson('eaton_patch1_buildings_final.geojson')
```
* **The action:** Export the non-overlapping, mathematically validated building outlines to a new GeoJSON file. This file serves as the final building inventory for subsequent damage and recovery assessments.

In [6]:
regularizer = BuildingRegularizer(batch_size=8)

final_buildings = regularizer(assets_with_images)
_ = final_buildings.to_geojson('eaton_patch1_buildings_final.geojson')

2026-05-27 07:27:41- INFO: Loading processor and weights for facebook/sam3 (this may take a minute)...


Loading weights:   0%|          | 0/1468 [00:00<?, ?it/s]

2026-05-27 07:27:47- INFO: SAM 3 model facebook/sam3 loaded successfully.
2026-05-27 07:27:47- INFO: Starting Building Regularization Pipeline...
2026-05-27 07:27:47- INFO: Step 1/4: Running SAM 3 segmentation on image crops...
2026-05-27 07:27:47- INFO: SAM 3: Segmenting 276 images across assets in batches of 8...


Segmenting Image Batches: 100%|█████████████████| 35/35 [01:12<00:00,  2.06s/it]

2026-05-27 07:29:00- INFO: SAM 3: All applicable images segmented successfully.
2026-05-27 07:29:00- INFO: Step 2/4: Georeferencing masks and dynamically bridging gaps...


2026-05-27 07:29:09- INFO: Step 3/4: Retrying 66 dropped assets with lower thresholds...
2026-05-27 07:29:09- INFO: SAM 3: Segmenting 66 images across assets in batches of 8...


Segmenting Image Batches: 100%|███████████████████| 9/9 [00:17<00:00,  1.91s/it]

2026-05-27 07:29:26- INFO: SAM 3: All applicable images segmented successfully.


2026-05-27 07:29:33- INFO: Pass 2 recovered 75 previously dropped assets.
2026-05-27 07:29:33- INFO: Step 4/4: Dissolving overlapping and adjoining polygons...
2026-05-27 07:29:34- INFO: Step 5: Truncating intrusions between neighboring polygons...
2026-05-27 07:29:34- INFO: Step 6: Filtering out degenerate geometries...
2026-05-27 07:29:34- INFO: Discarded 3 degenerate geometries during final cleanup.
2026-05-27 07:29:34- INFO: Updated attribute 'asset_type' for 0 assets.
2026-05-27 07:29:34- INFO: Building regularization complete.


## <u>Part 2: Aerial Post-Disaster Damage Assessment</u>
With the baseline building inventory established, we proceed to damage assessment. We will use the post-fire RAPID drone orthomosaic to evaluate each building using the Combustion Hierarchy Scale (CHS) with a local Vision-Language Model (VLM).

### **Workflow Overview:**

* **Configure the Pipeline:**
  The `Pipeline`  engine is initialized to systematically manage the progression of data from raw image extraction through to AI-based analysis.
* **Extract Targeted Imagery:**
  The `AerialImageryExtractor` segments high-resolution patches from the post-disaster drone map for each footprint in the inventory. A red bounding outline is added to each image to visually direct the AI model to the intended target building.
* **Local AI Evaluation:**
  The `Gemma4AssetAnalyzer` uses a local Gemma-4 vision-language model (VLM) to process cropped images and assign scores based on a custom CHS prompt. Optimized sequential GPU batching is employed to maximize throughput.
* **Filter and Export:**
  Assets that cannot be processed, such as those located outside the drone map boundaries, are removed. The AI-enriched data is then serialized to disk.

> **Result:** *A comprehensive GeoJSON inventory enriched with AI-generated CHS damage scores and justifications, primed for localized street-level recovery tracking.*

### Configure the extraction and analysis pipeline

**Downloading the damage classification prompts**
```python
[prompt_path] = download_dataset(['aerial_chs_prompts'])
```
* **The action:** We use `download_dataset` function to retrieve the text file that contains the Combustion Hierarchy Scale (CHS) definitions for aerial imagery.
* **Why download a file with prompts?:** By storing extensive text prompts in a dedicated file rather than embedding them directly in the Python script, we maintain cleaner code and enable researchers to efficiently version-control and modify AI instructions.

**Initializing the core pipeline**
```python
pipeline = Pipeline()
```
* **The action:** We  start by initializing the `rapidtools` `Pipeline` engine. This tool ensures each step runs in the correct order and automatically transfers data from the extractor to the AI analyzer.

**Configuring the aerial image extractor**
```python
extractor = AerialImageryExtractor(
    dataset=raster_path,
    save_directory='eaton_fire_aerial_feb25/overlaid_imagery',
    overlay_asset_outline=True,
    image_prefix='eaton_trinity_25',
    keep_multiple_copies=True,
)
```
* **The action:** The extractor is configured to generate high-resolution image patches from the UW RAPID post-disaster orthomosaic for each building outline in the inventory.
* **The visual anchor:** Setting `overlay_asset_outline=True` serves as an essential vision-language model prompting technique. This parameter draws a red boundary around the target building in the saved JPEG image. When multiple houses are present in the image crop, the red outline ensures that the AI model identifies the correct roof for evaluation.

**Configuring the local vision-language model**
```python
analyzer = Gemma4AssetAnalyzer(
    model_id='google/gemma-4-E2B-it',
    prompt=prompt_path,
    batch_size=8
)
```
* **The action:** A local instance of Google’s Gemma-4 2B model is configured to process the cro images and assign scores according to the custom CHS prompt.
* **The performance:** By setting `batch_size=8`,  the pipeline efficiently utilizes the GPU’s VRAM, enabling simultaneous analysis of 8 buildings and significantly reducing total inference time.

In [7]:
[prompt_path] = download_dataset('aerial_chs_prompts')

pipeline = Pipeline()

extractor = AerialImageryExtractor(
    dataset=raster_path,
    save_directory='eaton_fire_aerial_feb25/overlaid_imagery',
    overlay_asset_outline=True,
    image_prefix='eaton_trinity_25',
    keep_multiple_copies=True,
)

analyzer = Gemma4AssetAnalyzer(
    model_id='google/gemma-4-E2B-it',
    prompt=prompt_path,
    batch_size=8
)

2026-05-27 07:29:35- INFO: Preparing to download 1 file(s) across 1 dataset(s)...


2026-05-27 07:29:35- INFO: Successfully secured 1/1 files.
2026-05-27 07:29:35- INFO: Loading processor for google/gemma-4-E2B-it...


2026-05-27 07:29:45- INFO: Loading model google/gemma-4-E2B-it (this may take a while)...


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

### Execute the workflow and export results

**Running the multi-modal pipeline**
```python
pipeline.add_step(extractor)
pipeline.add_step(analyzer)

processed_collection = pipeline.run(final_buildings)
```
* **The action:** We register the configured tools within the pipeline and input the `final_buildings` inventory. The pipeline first extracts imagery, then streams these images directly into the Gemma-4 analyzer.

**Filtering and exporting the AI-enriched data**
```python
final_collection = processed_collection.filter_empty()

_ = final_collection.to_geojson(
    'eaton_footprints_CHS.geojson',
    ignore_properties=['image_assets']
)
```
* **The action:** Any assets that failed to process, such as footprints located outside the drone map boundaries, are removed from the collection
* **The output:** Exported dataset. The new AI-generated insights, such as `"gemma4_chs_level"`are now included in the GeoJSON attributes. The `image_assets` metadata is omitted to ensure the file remains lightweight and suitable for GIS applications.

In [8]:
pipeline.add_step(extractor)
pipeline.add_step(analyzer)

processed_collection = pipeline.run(final_buildings)

final_collection = processed_collection.filter_empty()

_ = final_collection.to_geojson(
    'eaton_footprints_CHS.geojson',
    ignore_properties=['image_assets']
)

2026-05-27 07:29:54- INFO: Starting pipeline with 2 steps...
2026-05-27 07:29:54- INFO: --- Running step 1/2: AerialImageryExtractor ---
2026-05-27 07:29:54- INFO: Found 1 raster(s). Images will be saved to: /home/bacetiner/Desktop/workshop_demo/eaton_fire_aerial_feb25/overlaid_imagery
2026-05-27 07:29:54- INFO: Processing raster: eaton_patch_20250214.tiff
2026-05-27 07:29:55- INFO: Found 181/181 assets within the current raster's extent.


Extracting from eaton_patch_20250214.tiff: 100%|█| 181/181 [00:19<00:00,  9.11it

2026-05-27 07:30:15- INFO: Finished processing all rasters. Extracted aerial imagery for a total of 181 assets.


2026-05-27 07:30:15- INFO: --- Running step 2/2: Gemma4AssetAnalyzer ---
2026-05-27 07:30:15- INFO: Analyzing 181 assets in batches of 8...


Batch Analysis: 100%|███████████████████████████| 23/23 [08:05<00:00, 21.12s/it]

2026-05-27 07:38:20- INFO: All assets processed successfully.
2026-05-27 07:38:20- INFO: Pipeline execution successfully completed.


## <u>Part 3: Multi-Temporal Street-Level Recovery Tracking</u>
While aerial imagery offers a comprehensive macro-level perspective on destruction, assessing physical recovery activities, including debris clearance, site scraping, and active framing, requires the detailed resolution of street-level panoramas. In the final step, the inventory is filtered to isolate heavily damaged buildings, and their reconstruction progress is tracked over several months using multi-temporal street view data.

### **Workflow Overview:**

* **Isolate Damaged Assets:**
  The dataset is filtered using standard Python list comprehensions to retain only those buildings previously classified by the AI as CHS Levels 1 through 4.
* **Extract Multi-Temporal Imagery:**
  The `MapillaryPanoramaExtractor` conducts targeted spatial queries to download the optimal unobstructed street-view image from both the immediate post-disaster period (February) and the recovery period (September).
* **Comparative AI Analysis:**
  The `Gemma4AssetAnalyzer` is re-deployed with a recovery-specific prompt. It processes the 'Before' and 'After' street-level images simultaneously, requiring the vision-language model (VLM) to perform a direct temporal comparison to identify site cleanup and new construction.

> **Result:** *A final, multi-modal dataset that documents the complete lifecycle of regional infrastructure, from pre-disaster baselines and post-event destruction to long-term community recovery.*

### Filter the asset inventory and prepare credentials

**Isolating severely damaged assets in a specific area**
```python
recovery_area = PolygonRegion([
    (-118.1426, 34.1851), 
    (-118.1450, 34.1859), 
    (-118.1450, 34.1878), 
    (-118.1396, 34.1879), 
    (-118.1396, 34.1862), 
    (-118.1418, 34.1869), 
    (-118.1426, 34.1851)
])

damaged_collection = final_collection.filter_by_geometry(recovery_area).filter_by_attribute(
    key='gemma4_chs_level', 
    value=['3', '4'], 
    operator='in'
)
```
* **The action:** We define a custom geographic boundary using `PolygonRegion` and apply two sequential filters. First, `filter_by_geometry` selects buildings located within the target neighborhood. Next, `filter_by_attribute` limits the dataset to severely damaged structures (CHS Levels 3 and 4). 

**Loading the API token and recovery prompt**
```python
[recovery_prompts, token_path] = download_dataset([
    'street_recovery_prompts',
    'mapillary_token'
])

MAPILLARY_TOKEN = Path(token_path).read_text().strip()
recovery_prompt = Path(recovery_prompts).read_text().strip()
```
* **The action:** The Mapillary API access token and the recovery-specific large language model (LLM) instructions are downloaded using the `download_dataset` method and loaded to memory for later use.

In [9]:
recovery_area = PolygonRegion([
    (-118.1426, 34.1851), 
    (-118.1450, 34.1859), 
    (-118.1450, 34.1878), 
    (-118.1396, 34.1879), 
    (-118.1396, 34.1862), 
    (-118.1418, 34.1869), 
    (-118.1426, 34.1851)
])

damaged_collection = final_collection.filter_by_geometry(recovery_area).filter_by_attribute(
    key='gemma4_chs_level', 
    value=['3', '4'], 
    operator='in'
)

[recovery_prompts, token_path] = download_dataset([
    'street_recovery_prompts',
    'mapillary_token'
])

MAPILLARY_TOKEN = Path(token_path).read_text().strip()
recovery_prompt = Path(recovery_prompts).read_text().strip()

2026-05-27 07:38:21- INFO: Preparing to download 2 file(s) across 2 dataset(s)...


2026-05-27 07:38:22- INFO: Successfully secured 2/2 files.


### Extract Multi-Temporal Street-Level Imagery

**Fetching post-disaster "before" images**
```python
feb_extractor = MapillaryImageExtractor(
    access_token=MAPILLARY_TOKEN,
    save_directory='output/street_feb',
    start_date='2025-01-01',
    end_date='2025-05-01',
    max_images_per_asset=1
)
damaged_collection = feb_extractor(damaged_collection)
```

**Fetching recovery "after" images**
```python
sep_extractor = MapillaryImageExtractor(
    access_token=MAPILLARY_TOKEN,
    save_directory='output/street_sep',
    start_date='2025-06-01',
    end_date='2025-12-01',
    max_images_per_asset=1
)
damaged_collection = sep_extractor(damaged_collection)
```
* **The action:** The `MapillaryImageExtractor` is deployed twice on the same asset collection.
* **The spatial-temporal control:** Spatial-temporal control is achieved by adjusting the `start_date` and `end_date` parameters. Setting `max_images_per_asset` to 1 ensures that the extractor retrieves the optimal unobstructed view of the property from February and another from September. Both images are then attached to the same  `PhysicalAsset` within collection.

In [10]:
feb_extractor = MapillaryImageExtractor(
    access_token=MAPILLARY_TOKEN,
    save_directory='output/street_feb',
    start_date='2025-01-01',
    end_date='2025-05-01',
    max_images_per_asset=1
)
damaged_collection = feb_extractor(damaged_collection)

sep_extractor = MapillaryImageExtractor(
    access_token=MAPILLARY_TOKEN,
    save_directory='output/street_sep',
    start_date='2025-06-01',
    end_date='2025-12-01',
    max_images_per_asset=1
)
damaged_collection = sep_extractor(damaged_collection)

2026-05-27 07:38:22- INFO: Fetching regional Mapillary metadata...
2026-05-27 07:38:22- INFO: Scanning 1 tiles for images...


2026-05-27 07:38:37- INFO: Scan complete. Found 57558 unique images.
2026-05-27 07:38:37- INFO: Building KDTree for panos and STRtree for footprints...
2026-05-27 07:38:37- INFO: Extracting panos using 10 threads...



Extracting Panoramas:   0%|                              | 0/27 [00:00<?, ?it/s]

2026-05-27 07:38:39- INFO: Downloading 1521737715533102 into memory...
2026-05-27 07:38:39- INFO: Downloading 747667081413289 into memory...
2026-05-27 07:38:39- INFO: Downloading 1240315441204040 into memory...
2026-05-27 07:38:40- INFO: Downloading 1119999836675352 into memory...
2026-05-27 07:38:40- INFO: Downloading 2196819244154397 into memory...
2026-05-27 07:38:40- INFO: Downloading 1110786164318280 into memory...
2026-05-27 07:38:40- INFO: Downloading 1226092389200053 into memory...
2026-05-27 07:38:40- INFO: Downloading 646675914668800 into memory...
2026-05-27 07:38:40- INFO: Downloading 695398096853302 into memory...
2026-05-27 07:38:40- INFO: Downloading 1689332665200725 into memory...


/home/bacetiner/anaconda3/envs/rapidtools/lib/python3.11/site-packages/PIL/Image.py:3432: DecompressionBombWarning: Image size (91179008 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
Extracting Panoramas:  26%|█████▋                | 7/27 [00:07<00:14,  1.42it/s]

...

2026-05-27 07:39:26- INFO: Skipping duplicate asset with ID: bldg_merged_86078cae_major_pos


Extracting Panoramas:  93%|███████████████████▍ | 25/27 [00:17<00:01,  1.84it/s]

2026-05-27 07:39:26- INFO: Skipping duplicate asset with ID: bldg_merged_0f1fc135_major_pos
2026-05-27 07:39:27- INFO: Skipping duplicate asset with ID: bldg_merged_49cf1943_major_neg


Extracting Panoramas: 100%|█████████████████████| 27/27 [00:18<00:00,  1.43it/s]

2026-05-27 07:39:27- INFO: Finished! Successfully extracted 25 multi-angle cropped panoramas.


### Execute the comparative AI analysis and export

**Reconfigure the vision-language model**
```python
analyzer.prompt = recovery_prompt
analyzer.batch_size = 2
analyzer.max_images_per_asset = 2
analyzer.image_filter = lambda img: 'street_feb' in str(img.path) or 'street_sep' in str(img.path)
```
* **The action:** Instead of initializing a new AI analyzer, the same Gemma-4 model loaded in Part 2 is reused.
* **The inner workings:** Reusing the model prevents PyTorch from reloading the 4GB weight file, thereby eliminating the risk of an Out-Of-Memory crash. The settings have been updated, and a `lambda` function is applied to ensure the AI receives the February and September images for direct comparison.

**Running the recovery assessment**
```python
recovery_collection = analyzer(damaged_collection)

final_recovery_collection = recovery_collection.filter_empty()

_ = final_recovery_collection.to_geojson('damaged_buildings_recovery_status.geojson', ignore_properties=['image_assets'])
```
* **The action:** The vision-language model analyzes the two temporal states of each building, generates a JSON comparison assessing cleanup and framing progress, and integrates this information into the assets. Any empty assets are removed before exporting the final, comprehensive dataset.

In [11]:
analyzer.prompt = recovery_prompt
analyzer.batch_size = 2
analyzer.max_images_per_asset = 2
analyzer.image_filter = lambda img: 'street_feb' in str(img.path) or 'street_sep' in str(img.path)

recovery_collection = analyzer(damaged_collection)

final_recovery_collection = recovery_collection.filter_empty()

_ = final_recovery_collection.to_geojson('damaged_buildings_recovery_status.geojson', ignore_properties=['image_assets'])

2026-05-27 07:39:27- INFO: Analyzing 27 assets in batches of 2...


Batch Analysis: 100%|███████████████████████████| 14/14 [05:58<00:00, 25.60s/it]

2026-05-27 07:45:26- ERROR: 2 assets failed to process.
